# 11. LIKE · BETWEEN · IN

*Module: 4 — Advanced querying · Companion summary: `lectures/11-like-between-in.md` · SQL engine: SQLite (in-memory)*

## What you will study

- find patterns with LIKE and the three wildcard positions
- treat LIKE's case sensitivity correctly in SQLite
- filter numeric ranges with BETWEEN
- filter lists of values with IN

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — BOOK

In [ ]:
q("CREATE TABLE book (book_id TEXT PRIMARY KEY, title TEXT, pages INTEGER, year INTEGER)")
q("""
INSERT INTO book VALUES
    ('B1', 'The Little Prince',   292, 1943),
    ('B2', 'Journey to the West', 298, 1592),
    ('B3', 'SQL Basics',          150, 2018),
    ('B4', 'Data Science Handbook', 400, 2020),
    ('B5', 'A Short Story',        45, 2022)
""")
q("SELECT * FROM book")

## 🧪 LIKE — pattern matching with % (any run of characters)

In [ ]:
q("SELECT title FROM book WHERE title LIKE '%Prince%'")   # contains
q("SELECT title FROM book WHERE title LIKE 'The%'")   # starts with
q("SELECT title FROM book WHERE title LIKE '%Prince'")   # ends with
q("SELECT title FROM book WHERE title LIKE '%Story%'")   # contains, later case


## 🧪 The second wildcard _ — exactly ONE character

In [ ]:
q("SELECT title FROM book WHERE title LIKE '_ Short%'")   # 'A Short Story'

## 🧪 LIKE + functions — case is something YOU control

SQLite's LIKE is case-insensitive for ASCII by default (MySQL too; DB2 is
case-sensitive), but the reliable habit is to normalize with UPPER()/LOWER()
*or* rely on SQLite's builtin — see both:

In [ ]:
q("SELECT title FROM book WHERE UPPER(title) LIKE '%LITTLE%'")   # explicit normalization
q("SELECT title FROM book WHERE title LIKE '%little%'")   # SQLite default: matches too


## 🧪 BETWEEN — inclusive numeric range

In [ ]:
q("SELECT title, pages FROM book WHERE pages BETWEEN 290 AND 300")
q("SELECT title, pages FROM book WHERE pages BETWEEN 150 AND 292")   # boundaries included

## 🧪 IN — membership in a list

In [ ]:
q("SELECT title FROM book WHERE book_id IN ('B1', 'B3', 'B5')")
q("SELECT title FROM book WHERE year IN (1943, 1592, 2020)")

## 💡 Tips & tricks

- `%` = any string (even empty), `_` = exactly one character — practice until instant.
- NOT LIKE / NOT BETWEEN / NOT IN all exist and read naturally.
- 📕 Dialect note: DB2/Oracle spell `BETWEEN ... AND` the same; for case-insensitive
  matching everywhere, always write `WHERE LOWER(col) LIKE '%x%'`.

## 🎯 Your turn

1. All titles containing the word 'the' (any position).
2. Books of 100–200 pages, using BETWEEN.
3. Titles of books NOT in years 2018 and 2020 (use NOT IN).

In [ ]:
# 💡 Solution
q("SELECT title FROM book WHERE title LIKE '%the%'")
q("SELECT title FROM book WHERE pages BETWEEN 100 AND 200")
q("SELECT title FROM book WHERE year NOT IN (2018, 2020)")